# Bab 07 · Objek dan Kelas Secukupnya

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Menjaga data tiap instance tetap mandiri.
- Menggunakan properti dan dataclass untuk kontrak objek.
- Membedakan tahap fit dan transform.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 9

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 9

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 9

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 9

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 9

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 9

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 9

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 9

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 9

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
from dataclasses import dataclass, field

## [BACA] Konsep inti

Atribut kelas dibagi oleh instance; atribut yang ditetapkan melalui `self` biasanya milik instance. Properti dapat menyajikan nilai turunan. Kesamaan nilai (`==`) berbeda dari identitas (`is`). Objek hashable harus memiliki hash yang konsisten dengan kesamaannya. Model sederhana dapat menyimpan hasil belajar pada `fit` untuk dipakai lagi pada `transform`.

## [DUGA] Prediksi sebelum eksekusi

Dua instance berbeda dibuat. Mengapa isi `b` dapat berubah?

**Prediksi saya:** …

**Alasan:** …

**Definisi `KeranjangContoh`** · bagian 1 dari 2

In [ ]:
# [COBA]
class KeranjangContoh:
    isi = []

    def tambah(self, nama):
        self.isi.append(nama)

**Jalankan contoh atau lanjutkan langkah sebelumnya** · bagian 2 dari 2

In [ ]:
a, b = KeranjangContoh(), KeranjangContoh()
a.tambah("Nastar")
print(b.isi)
print(a is b, a.isi is b.isi)

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Keranjang per instance

**[ISI KODE]**

Lengkapi kelas `Keranjang`: tiap objek mulai dengan list `isi` kosong miliknya sendiri. Metode `tambah(item)` menambahkan satu item. `len(objek)` mengembalikan banyak item.

> Petunjuk: Tentukan `self.isi` di konstruktor.

In [ ]:
# [ISI KODE]
class Keranjang:
    def __init__(self):
        raise BelumDiisi()

    def tambah(self, item):
        raise BelumDiisi()

    def __len__(self):
        raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    a, b = Keranjang(), Keranjang()
    a.tambah("A")
    sama(a.isi, ["A"])
    sama(b.isi, [])
    sama(len(a), 1)
    sama(len(b), 0)
    assert a.isi is not b.isi, "List tiap instance harus berbeda."

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · Transaksi dan properti total

**[ISI KODE]**

Buat `Transaksi(produk, jumlah, harga)`. Tolak jumlah atau harga negatif dengan `ValueError`. Simpan ketiga atribut, sediakan properti baca `total = jumlah × harga`, dan `repr` yang memuat nama produk. Input angka berhingga; tidak perlu setter.

> Petunjuk: Gunakan dekorator `@property`; akses total tanpa tanda kurung.

In [ ]:
# [ISI KODE]
class Transaksi:
    def __init__(self, produk, jumlah, harga):
        raise BelumDiisi()

    @property
    def total(self):
        raise BelumDiisi()

    def __repr__(self):
        raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    t = Transaksi("Nastar", 3, 85000)
    sama(t.total, 255000)
    assert "Nastar" in repr(t), "repr perlu memuat produk."
    sama(Transaksi("X", 0, 10).total, 0)
    harus_galat(ValueError, lambda: Transaksi("X", -1, 10))
    harus_galat(ValueError, lambda: Transaksi("X", 1, -10))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · Catatan tetap dan kesamaan

**[ISI KODE]**

Buat dataclass `Titik(x, y)` yang immutable (`frozen=True`). Dua titik dengan koordinat sama harus setara, dapat menjadi anggota set, dan tidak boleh berubah koordinatnya.

> Petunjuk: Pakai dekorator dataclass dengan opsi frozen dan deklarasi tipe atribut.

In [ ]:
# [ISI KODE]
# Ganti fungsi penanda ini dengan dataclass Titik.
def Titik(x, y):
    raise BelumDiisi()

**Definisi `uji_03`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_03():
    from dataclasses import FrozenInstanceError, is_dataclass

    a, b = Titik(1, 2), Titik(1, 2)
    assert is_dataclass(a), "Gunakan dataclass."
    assert a is not b and a == b, "Bedakan kesamaan dari identitas."
    sama(len({a, b, Titik(2, 1)}), 2)
    harus_galat(FrozenInstanceError, lambda: setattr(a, "x", 9))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(3, uji_03)

## Latihan 4 · Penskala yang belajar sekali

**[ISI KODE]**

Buat `PenskalaPusat`. `fit(data)` menerima list angka tidak kosong, menyimpan `rerata_`, dan mengembalikan `self`. `transform(data)` mengurangi setiap nilai dengan rerata hasil fit, tanpa menghitung ulang rerata. Transform sebelum fit memunculkan `RuntimeError`; fit kosong memunculkan `ValueError`.

> Petunjuk: Simpan hasil fit pada atribut instance; periksa atribut itu saat transform.

In [ ]:
# [ISI KODE]
class PenskalaPusat:
    def fit(self, data):
        raise BelumDiisi()

    def transform(self, data):
        raise BelumDiisi()

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    p = PenskalaPusat()
    harus_galat(RuntimeError, lambda: p.transform([1]))
    assert p.fit([2, 4, 6]) is p, "fit harus mengembalikan self."
    sama(p.transform([10, 12]), [6, 8])
    sama(p.transform([]), [])
    dekat(p.rerata_, 4)
    harus_galat(ValueError, lambda: PenskalaPusat().fit([]))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.